In [63]:
import pandas as pd

In [64]:
df = pd.read_csv("dataset.csv", engine='python')

In [65]:
df.columns

Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')

In [66]:
# Combine title + review text
df['text'] = df['Review Title'].fillna('') + " " + df['Review Text'].fillna('')

# Check result
print(df[['text']].head())

                                                text
0  A Store That Doesn't Want to Sell Anything I r...
1  Had multiple orders one turned up and… Had mul...
2  I informed these reprobates I informed these r...
3  Advertise one price then increase it on websit...
4  If I could give a lower rate I would If I coul...


In [67]:
df.head(5)

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience,text
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024",A Store That Doesn't Want to Sell Anything I r...
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and…,Had multiple orders one turned up and driver h...,"September 16, 2024",Had multiple orders one turned up and… Had mul...
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024",I informed these reprobates I informed these r...
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024",Advertise one price then increase it on websit...
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024",If I could give a lower rate I would If I coul...


In [68]:
# Extract number from rating text
# Extract rating number safely
df['Rating'] = df['Rating'].astype(str).str.extract(r'(\d+)')

# Convert to numeric (handle errors safely)
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

# Drop rows where rating couldn't be extracted
df = df.dropna(subset=['Rating'])

# Convert to int
df['Rating'] = df['Rating'].astype(int)

# Check
print(df['Rating'].head())
print(df['Rating'].dtype)

0    1
1    1
2    1
3    1
4    1
Name: Rating, dtype: int64
int64


In [69]:
def get_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

df['sentiment'] = df['Rating'].apply(get_sentiment)

# Check
print(df[['Rating', 'sentiment']].tail())

       Rating sentiment
21209       5  Positive
21210       5  Positive
21211       3   Neutral
21212       5  Positive
21213       4  Positive


In [70]:
# ===========================
# Issue Keywords
# ===========================

issue_keywords = {
    "Refund": [
        "refund",
        "return",
        "money back",
        "return my money",
        "refund request",
        "cancel order"
    ],

    "Delivery": [
        "delivery",
        "late",
        "delay",
        "delayed",
        "late delivery",
        "delivery delayed",
        "not delivered",
        "not been delivered"
        "still waiting",
        "never arrived",
        "order not received",
        "shipment delayed",
        "missing package"
    ],

    "Product Issue": [
        "broken",
        "damaged",
        "defective",
        "not working",
        "faulty",
        "cracked",
        "stopped working",
        "poor quality",
        "missing parts",
        "battery issue"
    ],

    "Account Issue": [
        "login",
        "account",
        "password",
        "sign in",
        "unable to login",
        "can't login",
        "account locked",
        "reset password"
    ],

    "Billing": [
        "payment",
        "charged",
        "charged twice",
        "double charged",
        "billing",
        "billing issue",
        "payment failed",
        "money deducted",
        "incorrect bill",
        "wrong amount",
        "extra charge"
    ]
}


# ===========================
# Issue Detection Function
# ===========================

def get_issue(text):

    text = str(text).lower()

    for issue, keywords in issue_keywords.items():
        if any(keyword in text for keyword in keywords):
            return issue

    return "General"


# Create Issue Column
df["issue"] = df["text"].apply(get_issue)

# Check Results
print(df[["text", "issue"]].head())

                                                text          issue
0  A Store That Doesn't Want to Sell Anything I r...  Account Issue
1  Had multiple orders one turned up and… Had mul...       Delivery
2  I informed these reprobates I informed these r...  Account Issue
3  Advertise one price then increase it on websit...        General
4  If I could give a lower rate I would If I coul...         Refund


In [71]:
# def get_priority(row):
#     sentiment = row['sentiment']
#     text = row['text'].lower()

#     # High priority conditions
#     if sentiment == "Negative" and (
#         "urgent" in text or "immediately" in text or "now" in text or "worst" in text
#     ):
#         return "High"

#     # Medium priority
#     elif sentiment == "Negative":
#         return "Medium"

#     # Neutral
#     elif sentiment == "Neutral":
#         return "Medium"

#     # Positive
#     else:
#         return "Low"


# df['priority'] = df.apply(get_priority, axis=1)

# # Check
# print(df[['sentiment', 'priority']].head())

In [72]:
def get_priority(row):

    sentiment = row["sentiment"]
    issue = row["issue"]
    text = str(row["text"]).lower()

    urgent_words = [
        "urgent",
        "immediately",
        "asap",
        "worst",
        "fraud",
        "legal",
        "lawsuit"
    ]

    if sentiment == "Positive":
        return "Low"

    if sentiment == "Neutral":
        return "Medium"

    # Negative reviews
    if issue == "Refund":
        return "High"

    if issue == "Billing":
        return "High"

    if any(word in text for word in urgent_words):
        return "High"

    return "Medium"


df["priority"] = df.apply(get_priority, axis=1)

In [73]:
# Input feature (customer reviews)
reviews = df['text']

# Multiple target columns
targets = df[['sentiment', 'issue', 'priority']]

In [74]:
from sklearn.model_selection import train_test_split

reviews_train, reviews_test, targets_train, targets_test = train_test_split(
    reviews,
    targets,
    test_size=0.2,
    random_state=42
)

In [75]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,3),
    stop_words="english",
    min_df=2
)

reviews_train = vectorizer.fit_transform(reviews_train)
reviews_test = vectorizer.transform(reviews_test)

In [76]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import LinearSVC

svc_model = MultiOutputClassifier(
    LinearSVC(class_weight='balanced')
)

svc_model.fit(reviews_train, targets_train)

# Predictions
svc_predictions = svc_model.predict(reviews_test)

In [77]:
from sklearn.metrics import classification_report

print("Sentiment Report")
print(classification_report(
    targets_test['sentiment'],
    svc_predictions[:, 0]
))

print("Issue Report")
print(classification_report(
    targets_test['issue'],
    svc_predictions[:, 1]
))

print("Priority Report")
print(classification_report(
    targets_test['priority'],
    svc_predictions[:, 2]
))

Sentiment Report
              precision    recall  f1-score   support

    Negative       0.94      0.95      0.95      2921
     Neutral       0.23      0.17      0.19       166
    Positive       0.87      0.89      0.88      1124

    accuracy                           0.90      4211
   macro avg       0.68      0.67      0.67      4211
weighted avg       0.90      0.90      0.90      4211

Issue Report
               precision    recall  f1-score   support

Account Issue       0.81      0.81      0.81       286
      Billing       0.56      0.45      0.50        49
     Delivery       0.92      0.87      0.90       964
      General       0.91      0.98      0.95      1713
Product Issue       0.67      0.57      0.62        72
       Refund       0.97      0.91      0.94      1127

     accuracy                           0.91      4211
    macro avg       0.81      0.77      0.78      4211
 weighted avg       0.91      0.91      0.91      4211

Priority Report
              precis

In [78]:
review = "Resetting my password didn't help. I still can't sign in."

review_vector = vectorizer.transform([review])
prediction = svc_model.predict(review_vector)

print(prediction)

[['Negative' 'Account Issue' 'Medium']]


In [79]:
import joblib

# Save model
joblib.dump(svc_model, 'linear_svc_model.pkl')

# Save vectorizer
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Files saved successfully!")

Files saved successfully!


In [80]:
from google.colab import files

# Download model
files.download('linear_svc_model.pkl')

# Download vectorizer
files.download('tfidf_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>